# Frequent Actor Groups in the IMDb Top 1000 Movies

**Algorithms for Massive Datasets** — Master in Data Science for Economics, Università degli Studi di Milano, A.Y. 2025/26

**Project:** market-basket analysis. Each movie is a *basket* and its four billed stars are the *items*. The goal is to find groups of actors that frequently appear together, using techniques from the course (RU chapter 6), and to study how these techniques scale with the size of the data.

### Methods
| Method | Reference | Role in this notebook |
|---|---|---|
| Exhaustive counting | — | Ground truth: counts every subset of every basket |
| A-Priori (basic) | RU §6.2 | Textbook algorithm, one pass per itemset size |
| A-Priori (optimized) | RU §6.2 | Same algorithm, cheaper candidate generation and counting |
| PCY | RU §6.3.1 | Hash-based filter that reduces the pairs counted in the second pass |

Every algorithm is checked against exhaustive counting on every run.

### Structure
0. Setup
1. Dataset and the parts considered
2. Data organisation
3. Pre-processing
4. Algorithms and implementations
5. Experimental setup
6. Results on the original data
7. Scalability
8. Discussion and conclusions

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/TheRealMamoot/amd-market-basket-imdb.git"
REPO_REF = "main"
IN_COLAB = "google.colab" in sys.modules


def git(*args, cwd):
    return subprocess.run(["git", *args], cwd=cwd, check=True, capture_output=True, text=True)


if IN_COLAB:
    PROJECT_ROOT = Path("/content/amd-market-basket-imdb")
    if not PROJECT_ROOT.exists():
        git("clone", "--quiet", REPO_URL, str(PROJECT_ROOT), cwd="/content")
    git("fetch", "--quiet", "--tags", "origin", cwd=PROJECT_ROOT)
    git("checkout", "--quiet", REPO_REF, cwd=PROJECT_ROOT)
    is_branch = subprocess.run(
        ["git", "show-ref", "--verify", "--quiet", f"refs/remotes/origin/{REPO_REF}"],
        cwd=PROJECT_ROOT,
    ).returncode == 0
    if is_branch:
        git("merge", "--quiet", "--ff-only", f"origin/{REPO_REF}", cwd=PROJECT_ROOT)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "-r", str(PROJECT_ROOT / "requirements.txt")],
        check=True,
    )
else:
    PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").is_dir():
    raise RuntimeError(f"Expected the repository root, found {PROJECT_ROOT}.")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

commit = git("rev-parse", "--short", "HEAD", cwd=PROJECT_ROOT).stdout.strip()
print(f"Environment: {'Colab' if IN_COLAB else 'local'}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Code version: {commit}")
print(f"Python: {sys.version.split()[0]}")

### Global settings

All parameters that control the experiments are defined once in the cell below and passed explicitly to the `src` functions in later sections, so every result can be reproduced by changing a single cell.

In [ ]:
# Seed for every random choice (e.g. subsampling the movies)
SEED = 42

# Fraction of the movies to use; 1.0 means the whole dataset
SAMPLE_FRACTION = 1.0

# Support thresholds, as numbers of movies, for the experiments on the original data
SUPPORTS = [1, 2, 3, 5, 10]

# Timed runs per configuration; runtimes are summarised by their median
REPEATS = 5

# Numbers of hash buckets tried for PCY
BUCKETS = [1_000, 10_000, 100_000, 1_000_000]

# Dataset sizes for the scalability experiments, as numbers of copies of the original data
COPY_COUNTS = [1, 10, 50, 100]

# Largest number of relabelled copies used for basic A-Priori, whose cost grows quadratically there
BASIC_MAX_COPIES = 10